In [ ]:
import os
import json
import pandas as pd
import numpy as np

# Benchmark experiment

In [ ]:
files = {
    "warehouse1": [
        # "replan_warehouse-20-40-10-2-1-random-12-k200_paths_2026-03-17_seed123_66delays"
    ],
    "maze1": [
        "tippingpoints_maze-128-128-1-even-1-k20_paths_2026-03-19_seed123",
        "tippingpoints_maze-128-128-1-even-2-k20_paths_2026-03-19_seed123",
        "tippingpoints_maze-128-128-1-even-3-k20_paths_2026-03-19_seed123"
    ]
}
all_results = {}
for map_name in files:
    all_results[map_name] = {}
    for filename in files[map_name]:
        scenario = filename.split("_")[1]
        filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "../output", map_name, f"{filename}.json")
        all_results[map_name][scenario] = json.load(open(filepath))

In [ ]:
df = pd.DataFrame(columns=["map", "Scenario", "Agents", "Search", "Gen.", "Paths", "Flex.", "Tip"
    # "avg_search_time_maeder", "avg_generation_time_maeder", "avg_found_paths_maeder", "avg_delay_improvement"
    ])
num_rows = 0
for map_name in all_results:
    for scenario in all_results[map_name]:
        maeder = {"search": [], "generation": [], "paths": [], "flex-paths": [],  "delays": {a: [] for a in all_results[map_name][scenario]}}
        flexsipp = {"search": [], "generation": [], "paths": [], "flex-paths": [], "found_tip": [], "delays": {a: [] for a in all_results[map_name][scenario]}}
        for delay_agent in all_results[map_name][scenario]:
            if "FlexSIPP" in all_results[map_name][scenario][delay_agent] and "Search Time" in all_results[map_name][scenario][delay_agent]["FlexSIPP"]:
                flexsipp["search"].append(all_results[map_name][scenario][delay_agent]["FlexSIPP"]["Search Time"])
                flexsipp["generation"].append(all_results[map_name][scenario][delay_agent]["FlexSIPP"]["gen_time"])
                flexsipp["paths"].append(len(all_results[map_name][scenario][delay_agent]["FlexSIPP"]["unique_routes_safe"]))
                flexsipp["flex-paths"].append(0)
                flexsipp["found_tip"].append(False)
                for path, atf_strings in all_results[map_name][scenario][delay_agent]["FlexSIPP"]["unique_routes_safe"].items():
                    for atf_str in atf_strings:
                        atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                        current_delay = (atf[1] + atf[3]) - all_results[map_name][scenario][delay_agent]["original_arrival_time"]
                        for t, t_atf, t_path, other_delays in all_results[map_name][scenario][delay_agent]["FlexSIPP"]["tipping_points"]:
                            if t_path == path:
                                current_delay += sum([min([d for x, d in other_delays[a].items()]) if other_delays[a] else 0 for a in other_delays])
                                flexsipp["flex-paths"][-1] += 1
                                flexsipp["found_tip"][-1] = True
                        flexsipp["delays"][delay_agent].append(current_delay)
                # maeder["search"].append(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["Search Time"])
                # maeder["generation"].append(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["gen_time"])
                # maeder["paths"].append(len(all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["unique_routes_safe"]))
                # for path, atf_strings in all_results[map_name][num_agents][delay_agent]["@MAEDeR"]["unique_routes_safe"].items():
                #     for atf_str in atf_strings:
                #         atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                #         current_delay = (atf[1] + atf[3]) - all_results[map_name][num_agents][delay_agent]["original_arrival_time"]
                #         maeder["delays"][delay_agent].append(current_delay)
        df.loc[num_rows] = [
            map_name,
            map_name + "-" + scenario.split("-")[-2], 
            len(flexsipp["search"]), 
            (sum(flexsipp["search"]) / len(flexsipp["search"])) / 1000, #search time in milliseconds from cpp 
            sum(flexsipp["generation"]) / len(flexsipp["generation"]), 
            sum(flexsipp["paths"]) / len(flexsipp["paths"]),
            sum(flexsipp["flex-paths"]) / len(flexsipp["flex-paths"]),
            sum(flexsipp["found_tip"])
            # sum(maeder["search"]) / len(maeder["search"]), 
            # sum(maeder["generation"]) / len(maeder["generation"]), 
            # sum(maeder["paths"]) / len(maeder["paths"]),
            # sum([min(maeder["delays"][a] if maeder["delays"][a] else [0]) - min(flexsipp["delays"][a]  if flexsipp["delays"][a] else [0]) for a in flexsipp["delays"]]) / len(flexsipp["delays"])
        ]
        num_rows += 1
df

In [ ]:
table_times = os.path.join(os.path.dirname(os.path.abspath("__file__")), "../output", "table_flexsipp_times.tex")
with open(table_times, "w") as f:
    latex = df.to_latex(index=False, columns=["Scenario", "Agents", "Search", "Gen.", "Paths", "Flex.", "Tip"], float_format="%.2f", caption="FlexSIPP results for finding any-start-time plans. Shows the number of agents over which the average values are given for the search time in seconds, the generation time in seconds, the number of paths found, the number of paths which use flexibility, and the number of agents for which a tipping point was found.", label="tab:times", position="t")
    f.write(latex)

# Replanning experiment


In [ ]:
replan_files = {
    "warehouse1": [
        # "replan_FlexSIPP_warehouse1_2026-03-18_seed123",
        # "replan_@MAEDeR_warehouse1_2026-03-18_seed123"
    ],
    "maze1": [
        "replan_FlexSIPP_maze1_2026-03-19_seed123",
        "replan_@MAEDeR_maze1_2026-03-19_seed123"
    ]
}
replan_results = {}
for map_name in replan_files:
    replan_results[map_name] = {}
    for file in replan_files[map_name]:
        algorithm = file.split("_")[1]
        filepath = os.path.join(os.path.dirname(os.path.abspath("__file__")), "../output", map_name, f"{file}.json")
        obj = json.load(open(filepath))
        for scen, data in obj.items():
            if scen in replan_results[map_name]:
                replan_results[map_name][scen][algorithm] = data
            else:
                replan_results[map_name][scen]= {algorithm: data}

In [ ]:
df = pd.DataFrame(columns=["map", "scenario", "num_delays", "num_agents_tested_flexsipp", "num_agents_test_maeder", "avg_search_time_flexsipp", "avg_preprocess_time_flexsipp", "avg_postprocess_time_flexsipp", "avg_found_paths_flexsipp", "avg_search_time_maeder", "avg_preprocess_time_maeder", "avg_postprocess_time_maeder", "avg_found_paths_maeder", "avg_delay_improvement"])
delay_improvement = pd.DataFrame(columns=["map", "scenario", "delay", "delay_agent", "original_arrival_time_flexsipp", "original_arrival_time_maeder", "delay_flexsipp", "delay_maeder", "delta_flexsipp", "delta_maeder"])
num_rows = 0
delay_rows = 0
plot_arrival_times = pd.DataFrame(columns=["map", "scenario", "delay_idx", "delay_agent", "original_arrival_time", "arrival_time_maeder", "arrival_time_flexsipp", "flexibility_used"])
for map_name in replan_results:
    for scen_file in replan_results[map_name]:
        maeder = {"search": [], "preprocess": [], "postprocess": [], "paths": [], "not_found": 0, "delays": {a: [] for a in replan_results[map_name][scen_file]["@MAEDeR"]}}
        flexsipp = {"search": [], "preprocess": [], "postprocess": [], "paths": [], "not_found":0, "delays": {a: [] for a in replan_results[map_name][scen_file]["FlexSIPP"]}}
        num_delays = len(replan_results[map_name][scen_file]["FlexSIPP"])
        if "FlexSIPP" in replan_results[map_name][scen_file]:
            for delay in replan_results[map_name][scen_file]["FlexSIPP"]:
                flex_delay = []
                flex_used = 0
                if not replan_results[map_name][scen_file]["FlexSIPP"][delay]:
                    print("No results for", map_name, scen_file, delay, "flexsipp")
                    flexsipp["not_found"] += 1
                else:
                    flexsipp["search"].append(replan_results[map_name][scen_file]["FlexSIPP"][delay]["Search Time"])
                    flexsipp["preprocess"].append(replan_results[map_name][scen_file]["FlexSIPP"][delay]["preprocess_time"])
                    flexsipp["postprocess"].append(replan_results[map_name][scen_file]["FlexSIPP"][delay]["postprocess_time"])
                    flexsipp["paths"].append(len(replan_results[map_name][scen_file]["FlexSIPP"][delay]["unique_routes_safe"]))
                    for path, atf_strings in replan_results[map_name][scen_file]["FlexSIPP"][delay]["unique_routes_safe"].items():
                        for atf_str in atf_strings:
                            atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                            current_delay = (atf[1] + atf[3]) - replan_results[map_name][scen_file]["FlexSIPP"][delay]["original_arrival_time"]
                            for t, t_atf, t_path, other_delays in replan_results[map_name][scen_file]["FlexSIPP"][delay]["tipping_points"]:
                                if t_path == path:
                                    current_delay += sum([min([d for x, d in other_delays[a].items()]) if other_delays[a] else 0 for a in other_delays])
                                    flex_used += sum([min([d for x, d in other_delays[a].items()]) if other_delays[a] else 0 for a in other_delays])
                            flexsipp["delays"][delay].append(current_delay)
                            flex_delay = [replan_results[map_name][scen_file]["FlexSIPP"][delay]["original_arrival_time"], current_delay, atf[3], atf[1]]
                if not replan_results[map_name][scen_file]["@MAEDeR"][delay]:
                    print("No results for", map_name, scen_file, delay, "maeder")
                    maeder["not_found"] += 1 
                else:
                    maeder["search"].append(replan_results[map_name][scen_file]["@MAEDeR"][delay]["Search Time"])
                    maeder["preprocess"].append(replan_results[map_name][scen_file]["@MAEDeR"][delay]["preprocess_time"])
                    maeder["postprocess"].append(replan_results[map_name][scen_file]["@MAEDeR"][delay]["postprocess_time"])
                    maeder["paths"].append(len(replan_results[map_name][scen_file]["@MAEDeR"][delay]["unique_routes_safe"]))
                    for path, atf_strings in replan_results[map_name][scen_file]["@MAEDeR"][delay]["unique_routes_safe"].items():
                        for atf_str in atf_strings:
                            atf = [float(x) if "inf" not in x else np.inf for x in atf_str.replace("[", "").replace("]", "").split(", ")]
                            current_delay = (atf[1] + atf[3]) - replan_results[map_name][scen_file]["@MAEDeR"][delay]["original_arrival_time"]
                            maeder["delays"][delay].append(current_delay)
                            if flex_delay:
                                delay_improvement.loc[delay_rows] = [
                                    map_name,
                                    scen_file,
                                    delay,
                                    replan_results[map_name][scen_file]["@MAEDeR"][delay]["delay_agent"],
                                    flex_delay[0],
                                    replan_results[map_name][scen_file]["@MAEDeR"][delay_agent]["original_arrival_time"],
                                    flex_delay[1],
                                    current_delay,
                                    flex_delay[2],
                                    atf[3]
                                ]
                                plot_arrival_times.loc[delay_rows] = [
                                    map_name,
                                    scen_file,
                                    delay,
                                    replan_results[map_name][scen_file]["@MAEDeR"][delay]["delay_agent"],
                                    replan_results[map_name][scen_file]["@MAEDeR"][delay]["original_arrival_time"],
                                    atf[1] + atf[3],
                                    flex_delay[2] + flex_delay[3],
                                    flex_used
                                ]
                                delay_rows += 1
            # traversal_times_maeder = [times["arrival"][1] - times["departure"][1] for agent, times in replan_results[map_name][scen_file]["@MAEDeR"]["delay0"].items()]
            # traversal_times_flexsipp = [times["arrival"][1] - times["departure"][1] for agent, times in replan_results[map_name][scen_file]["FlexSIPP"]["delay0"].items()]
            print(scen_file, replan_results[map_name][scen_file]["FlexSIPP"])
            diff_arrival_maeder = [replan_results[map_name][scen_file]["@MAEDeR"][delay_agent]["final_paths"][a]["arrival"][1] - replan_results[map_name][scen_file]["@MAEDeR"]["delay0"]["initial_paths"][a]["arrival"][1] for a in replan_results[map_name][scen_file]["@MAEDeR"]["delay0"]["initial_paths"]]
            # diff_arrival_flexsipp = [replan_results[map_name][scen_file]["FlexSIPP"][delay_agent]["final_paths"][a]["arrival"][1] - replan_results[map_name][scen_file]["FlexSIPP"]["delay0"]["initial_paths"][a]["arrival"][1] for a in replan_results[map_name][scen_file]["FlexSIPP"]["delay0"]["initial_paths"]]
            # print("maeder  ", diff_arrival_maeder, "\nflexsipp", diff_arrival_flexsipp)

            # df.loc[num_rows] = [
            #     map_name, 
            #     scen_file,
            #     num_delays,
            #     num_delays - flexsipp["not_found"],
            #     num_delays - maeder["not_found"],
            #     sum(flexsipp["search"]) / len(flexsipp["search"]), 
            #     sum(flexsipp["preprocess"]) / len(flexsipp["preprocess"]),
            #     sum(flexsipp["postprocess"]) / len(flexsipp["postprocess"]),
            #     sum(flexsipp["paths"]) / len(flexsipp["paths"]), 
            #     sum(maeder["search"]) / len(maeder["search"]), 
            #     sum(maeder["preprocess"]) / len(maeder["preprocess"]),
            #     sum(maeder["postprocess"]) / len(maeder["postprocess"]),
            #     sum(maeder["paths"]) / len(maeder["paths"]),
            #     sum([min(maeder["delays"][a] if maeder["delays"][a] else [0]) - min(flexsipp["delays"][a]  if flexsipp["delays"][a] else [0]) for a in flexsipp["delays"]]) / len(flexsipp["delays"])
            # ]
            # num_rows += 1
plot_arrival_times

In [ ]:
delay_improvement

In [ ]:
df = pd.DataFrame(columns=["map", "scenario", "num_delays", "num_agents_tested_flexsipp", "num_agents_test_maeder", "avg_search_time_flexsipp", "avg_preprocess_time_flexsipp", "avg_postprocess_time_flexsipp", "avg_found_paths_flexsipp", "avg_search_time_maeder", "avg_preprocess_time_maeder", "avg_postprocess_time_maeder", "avg_found_paths_maeder", "avg_delay_improvement"])
delay_improvement = pd.DataFrame(columns=["map", "scenario", "delay", "delay_agent", "original_arrival_time_flexsipp", "original_arrival_time_maeder", "delay_flexsipp", "delay_maeder", "delta_flexsipp", "delta_maeder"])
num_rows = 0
delay_rows = 0
plot_arrival_times = pd.DataFrame(columns=["map", "scenario", "delay_idx", "delay_agent", "original_arrival_time", "arrival_time_maeder", "arrival_time_flexsipp", "flexibility_used"])
for map_name in replan_results:
    for scen_file in replan_results[map_name]:
        maeder = {"search": [], "preprocess": [], "postprocess": [], "paths": [], "not_found": 0, "delays": {a: [] for a in replan_results[map_name][scen_file]["@MAEDeR"]}}
        flexsipp = {"search": [], "preprocess": [], "postprocess": [], "paths": [], "not_found":0, "delays": {a: [] for a in replan_results[map_name][scen_file]["FlexSIPP"]}}
        num_delays = len(replan_results[map_name][scen_file]["FlexSIPP"])
        if "FlexSIPP" in replan_results[map_name][scen_file]:
            for delay in replan_results[map_name][scen_file]["FlexSIPP"]:
                print(replan_results[map_name][scen_file]["FlexSIPP"][delay]["delay_agent"],replan_results[map_name][scen_file]["FlexSIPP"][delay]["original_arrival_time"],)
                print(replan_results[map_name][scen_file]["@MAEDeR"][delay]["delay_agent"],replan_results[map_name][scen_file]["@MAEDeR"][delay]["original_arrival_time"],)
